In [0]:
"""
id: python_loop_8f3a1c7e
template: python
templateVersion: 1.0.0
name: Process all customer tables
position:
  x: 0
  y: 0
description:
  text: Clean customer data by validating fields and formatting phone numbers. Separate valid and invalid records. Save results to target tables and flag reasons for bad records.
  hash: f26931e3
previewCodeHash: 25d55f45537510bd
codegenHash: 11228db4bc20a97b
updatedAt: 1792947152540
config:
  code: |
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, StructField, StringType
    from functools import reduce

    # Assign result immediately so the codegen default is never used
    result = spark.createDataFrame([], StructType([StructField("message", StringType())]))

    is_preview = config.get("meta_state", {}).get("is_preview", False)

    source_tables = [s.strip() for s in dbutils.widgets.get("source_table").split(",")]
    target_tables = [s.strip() for s in dbutils.widgets.get("target_table_name").split(",")]
    bad_tables = [s.strip() for s in dbutils.widgets.get("bad_records_table_name").split(",")]
    catalog = dbutils.widgets.get("target_catalog").strip()
    schema = dbutils.widgets.get("target_schema").strip()

    email_pattern = r'^[^@]+@[^@]+\.[^@]+$'
    not_null_cols = ["customer_id", "first_name", "last_name", "email", "phone"]
    fill_cols = ["first_name", "last_name", "email", "phone", "address", "country",
                 "age", "gender", "status", "loyalty_tier", "registration_date", "acquisition_channel"]

    all_good = []

    for i, src in enumerate(source_tables):
        target_name = target_tables[i] if i < len(target_tables) else f"{src.split('.')[-1]}_clean"
        bad_name = bad_tables[i] if i < len(bad_tables) else f"bad_records_{src.split('.')[-1]}"

        df = spark.table(src)

        fail_cond = F.lit(False)
        for c in not_null_cols:
            fail_cond = fail_cond | F.col(c).isNull()
        fail_cond = fail_cond | (~F.col("email").rlike(email_pattern))

        good_df = df.filter(~fail_cond)
        bad_df = df.filter(fail_cond)

        reason_parts = [F.when(F.col(c).isNull(), F.lit(f"{c} is missing")) for c in not_null_cols]
        reason_parts.append(F.when(~F.col("email").rlike(email_pattern), F.lit("email is invalid")))
        bad_df = bad_df.withColumn("quality_check_reason", F.concat_ws("; ", *reason_parts))

        good_df = good_df.withColumn(
            "country",
            F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""),
                F.when(F.upper(F.col("customer_id")).like("IND%"), F.lit("India"))
                 .when(F.upper(F.col("customer_id")).like("USA%"), F.lit("United States"))
                 .when(F.upper(F.col("customer_id")).like("UK%"), F.lit("United Kingdom"))
                 .otherwise(F.col("country"))
            ).otherwise(F.col("country"))
        )
        good_df = good_df.withColumn(
            "phone",
            F.when(F.upper(F.col("customer_id")).like("IND%"),
                F.concat(F.lit("+91 "), F.regexp_replace(F.regexp_replace(F.col("phone"), r"^(\+91|091|91)\s*", ""), r"(\d{5})(\d{5})", "$1 $2"))
            ).when(F.upper(F.col("customer_id")).like("USA%"),
                F.concat(F.lit("+1 "), F.regexp_replace(F.regexp_replace(F.col("phone"), r"^(\+1|1)\s*", ""), r"(\d{3})(\d{3})(\d{4})", "($1) $2-$3"))
            ).when(F.upper(F.col("customer_id")).like("UK%"),
                F.concat(F.lit("+44 "), F.regexp_replace(F.regexp_replace(F.col("phone"), r"^(\+44|044|44|0)\s*", ""), r"(\d{4})(\d{6})", "$1 $2"))
            ).otherwise(F.col("phone"))
        )
        good_df = good_df.withColumn("gender", F.when(F.col("gender").isNull() | (F.trim(F.col("gender")) == ""), F.lit("N")).otherwise(F.col("gender")))
        good_df = good_df.withColumn("acquisition_channel", F.when(F.col("acquisition_channel").isNull() | (F.trim(F.col("acquisition_channel")) == ""), F.lit("Unknown")).otherwise(F.col("acquisition_channel")))
        for c in fill_cols:
            good_df = good_df.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), F.lit("N/A")).otherwise(F.col(c)))

        if not is_preview:
            good_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.{target_name}")
            bad_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.{bad_name}")

        all_good.append(good_df.withColumn("_source_table", F.lit(src)))

    if all_good:
        result = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), all_good)
checkpoint:
  mode: auto
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, StructField, StringType
    from functools import reduce

    # Assign result immediately so the codegen default is never used
    result = spark.createDataFrame([], StructType([StructField("message", StringType())]))

    is_preview = config.get("meta_state", {}).get("is_preview", False)

    source_tables = [s.strip() for s in dbutils.widgets.get("source_table").split(",")]
    target_tables = [s.strip() for s in dbutils.widgets.get("target_table_name").split(",")]
    bad_tables = [s.strip() for s in dbutils.widgets.get("bad_records_table_name").split(",")]
    catalog = dbutils.widgets.get("target_catalog").strip()
    schema = dbutils.widgets.get("target_schema").strip()

    email_pattern = r'^[^@]+@[^@]+\.[^@]+$'
    not_null_cols = ["customer_id", "first_name", "last_name", "email", "phone"]
    fill_cols = ["first_name", "last_name", "email", "phone", "address", "country",
                 "age", "gender", "status", "loyalty_tier", "registration_date", "acquisition_channel"]

    all_good = []

    for i, src in enumerate(source_tables):
        target_name = target_tables[i] if i < len(target_tables) else f"{src.split('.')[-1]}_clean"
        bad_name = bad_tables[i] if i < len(bad_tables) else f"bad_records_{src.split('.')[-1]}"

        df = spark.table(src)

        fail_cond = F.lit(False)
        for c in not_null_cols:
            fail_cond = fail_cond | F.col(c).isNull()
        fail_cond = fail_cond | (~F.col("email").rlike(email_pattern))

        good_df = df.filter(~fail_cond)
        bad_df = df.filter(fail_cond)

        reason_parts = [F.when(F.col(c).isNull(), F.lit(f"{c} is missing")) for c in not_null_cols]
        reason_parts.append(F.when(~F.col("email").rlike(email_pattern), F.lit("email is invalid")))
        bad_df = bad_df.withColumn("quality_check_reason", F.concat_ws("; ", *reason_parts))

        good_df = good_df.withColumn(
            "country",
            F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""),
                F.when(F.upper(F.col("customer_id")).like("IND%"), F.lit("India"))
                 .when(F.upper(F.col("customer_id")).like("USA%"), F.lit("United States"))
                 .when(F.upper(F.col("customer_id")).like("UK%"), F.lit("United Kingdom"))
                 .otherwise(F.col("country"))
            ).otherwise(F.col("country"))
        )
        good_df = good_df.withColumn(
            "phone",
            F.when(F.upper(F.col("customer_id")).like("IND%"),
                F.concat(F.lit("+91 "), F.regexp_replace(F.regexp_replace(F.col("phone"), r"^(\+91|091|91)\s*", ""), r"(\d{5})(\d{5})", "$1 $2"))
            ).when(F.upper(F.col("customer_id")).like("USA%"),
                F.concat(F.lit("+1 "), F.regexp_replace(F.regexp_replace(F.col("phone"), r"^(\+1|1)\s*", ""), r"(\d{3})(\d{3})(\d{4})", "($1) $2-$3"))
            ).when(F.upper(F.col("customer_id")).like("UK%"),
                F.concat(F.lit("+44 "), F.regexp_replace(F.regexp_replace(F.col("phone"), r"^(\+44|044|44|0)\s*", ""), r"(\d{4})(\d{6})", "$1 $2"))
            ).otherwise(F.col("phone"))
        )
        good_df = good_df.withColumn("gender", F.when(F.col("gender").isNull() | (F.trim(F.col("gender")) == ""), F.lit("N")).otherwise(F.col("gender")))
        good_df = good_df.withColumn("acquisition_channel", F.when(F.col("acquisition_channel").isNull() | (F.trim(F.col("acquisition_channel")) == ""), F.lit("Unknown")).otherwise(F.col("acquisition_channel")))
        for c in fill_cols:
            good_df = good_df.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), F.lit("N/A")).otherwise(F.col(c)))

        if not is_preview:
            good_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.{target_name}")
            bad_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.{bad_name}")

        all_good.append(good_df.withColumn("_source_table", F.lit(src)))

    if all_good:
        result = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), all_good)

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["python_loop_8f3a1c7e.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "python_loop_8f3a1c7e" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["python_loop_8f3a1c7e.result"])
if "python_loop_8f3a1c7e" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "python_loop_8f3a1c7e", "counts": {"result": int(ctx["python_loop_8f3a1c7e.result"].count())}}}, raw=True)

---
id: note_summary_f1a2b3c4
template: markdown
name: Pipeline info
position:
  x: 269.2909822629315
  y: -212.14409500360176
dimensions:
  width: 480
  height: 290
config:
  md: |
    # Parameterized Customer Quality Pipeline

    The Python operator loops through **all customer tables** in the parameters:

    - **source_table**: comma-separated fully qualified source tables
    - **target_table_name**: comma-separated clean output table names
    - **bad_records_table_name**: comma-separated bad records table names
    - **target_catalog** / **target_schema**: shared catalog and schema

    **For each table it:**
    1. Checks customer_id, first/last name, email, phone not null
    2. Validates email format
    3. Cleanses good records (country, phone, gender, channel)
    4. Adds `quality_check_reason` to bad records
    5. Writes good and bad records to their target tables
---